# Proyecto ML 2025-2026 — Documentos Desclasificados del 23-F

**Notebook orquestador** — toda la lógica vive en `src/`.

| Caso | Clase | Descripción |
|------|-------|-------------|
| 1 | `EDACorpus` | Radiografía del corpus (EDA + clustering TF-IDF) |
| 2 | `ActorGraphCaso` | Grafo de co-menciones de actores |
| 3 | `TopicModelCaso` | Topic modeling semántico (BERTopic / NMF) |
| 4 | `ClasificadorCaso` | Clasificador supervisado de ministerio |
| 5 | `SpatioTemporalCaso` | Serie temporal + anomalías + geografía |

## 0. Setup

In [1]:
import logging
import sys
from pathlib import Path

# Añadir raíz del proyecto al path para importar src/
ROOT = Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s — %(message)s",
    datefmt="%H:%M:%S",
)
print(f"Raíz del proyecto: {ROOT}")

Raíz del proyecto: /Users/NachoATM_1/Documents/Estudios/MasterUNAV/Machine Learning/machine_learning


## 1. Obtención de datos

In [2]:
from src.data.scraper import Scraper23F

scraper = Scraper23F(
    base_url="https://23fbuscador.rtve.es",
    raw_path="data/documentos_23f_raw.json",
    delay=0.4,
)
raw = scraper.scrape(force_refresh=False)
print(f"Documentos cargados: {len(raw)}")

18:43:46 [src.data.scraper] INFO — Cargando corpus desde caché: data/documentos_23f_raw.json


Documentos cargados: 167


## 2. Construcción del corpus

In [3]:
from src.data.builder import CorpusBuilder

builder = CorpusBuilder(raw)
df = builder.build()

print(f"Shape: {df.shape}")
df[["id","titulo","fuente","anio","periodo","paginas",
    "n_personas","n_palabras_ocr","riqueza_lexica"]].head(5)

18:43:47 [src.data.builder] INFO — DataFrame construido: 167 docs - 19 variables


Shape: (167, 19)


,id,titulo,fuente,anio,periodo,paginas,n_personas,n_palabras_ocr,riqueza_lexica
0,1860,Vista oral 2/81 del Consejo Supremo de Justici...,Defensa,1982.0,Proceso judicial,3,10,640,0.6198
1,1859,Vista oral 2/81 del Consejo Supremo de Justici...,Defensa,1982.0,Proceso judicial,4,10,1018,0.4658
2,1858,Vista oral 2/81 del Consejo Supremo de Justici...,Interior,1982.0,Proceso judicial,5,10,1347,0.5258
3,1857,Vista oral 2/81 del Consejo Supremo de Justici...,Defensa,1982.0,Proceso judicial,6,10,1826,0.4628
4,1856,Vista oral 2/81 del Consejo Supremo de Justici...,Defensa,1982.0,Proceso judicial,6,10,1740,0.4328


In [4]:
# Vista rápida de la distribución
print(df["fuente"].value_counts().to_string())
print()
print(df["periodo"].value_counts().to_string())

fuente
Defensa           81
Interior          59
Exteriores        15
No determinado    12

periodo
Desconocido         78
Proceso judicial    64
23-F (1981)         21
Pre-golpe            3
Post-proceso         1


## 3. Caso 1 — EDA y Clustering del Corpus

In [5]:
from src.casos.caso1_eda import EDACorpus

caso1 = EDACorpus(df, output_dir="outputs", fig_dir="figuras")
results1 = caso1.run()
caso1.export()
print(caso1.summary())

18:43:48 [src.casos.caso1_eda] INFO — === Caso 1: EDA Corpus ===
18:43:48 [EDACorpus] INFO — Figura guardada: figuras/fig1a_fuente.png
18:43:48 [EDACorpus] INFO — Figura guardada: figuras/fig1b_paginas.png
18:43:48 [EDACorpus] INFO — Figura guardada: figuras/fig1c_periodo.png
18:43:48 [EDACorpus] INFO — Figura guardada: figuras/fig2_extension_riqueza.png
18:43:48 [EDACorpus] INFO — Figura guardada: figuras/fig3_timeline.png
18:43:49 [EDACorpus] INFO — Figura guardada: figuras/fig4a_personas_top.png
18:43:49 [EDACorpus] INFO — Figura guardada: figuras/fig4b_lugares_wordcloud.png
18:43:49 [EDACorpus] INFO — Figura guardada: figuras/fig5_actores_periodo.png
18:43:49 [EDACorpus] INFO — Figura guardada: figuras/fig6_wordclouds_periodo.png
18:43:50 [src.casos.caso1_eda] INFO — Varianza PCA(50): 61.9%
18:43:50 [EDACorpus] INFO — Figura guardada: figuras/fig7_elbow_silhouette.png
18:43:50 [EDACorpus] INFO — Figura guardada: figuras/fig8_pca2d_clusters.png
18:43:50 [EDACorpus] INFO — Figura gua

=== EDACorpus ===
  total_documentos: 167
  documentos_con_ocr: 167
  total_palabras: 346599
  ministerio_predominante: Defensa
  periodo_dominante: Desconocido
  persona_top: No Consta
  lugar_top: Madrid
  clusters_k: 7


## 4. Caso 2 — Grafo de Actores

In [6]:
from src.casos.caso2_grafos import ActorGraphCaso

caso2 = ActorGraphCaso(df, output_dir="outputs", fig_dir="figuras")
results2 = caso2.run()
caso2.export()
print(caso2.summary())

18:43:50 [src.casos.caso2_grafos] INFO — spaCy no disponible — usando columna 'personas' de RTVE.
18:43:50 [src.casos.caso2_grafos] INFO — === Caso 2: Grafo de Actores ===
18:43:50 [src.casos.caso2_grafos] INFO — Grafo: 936 nodos, 4223 aristas
18:43:53 [src.casos.caso2_grafos] INFO — Top 5 por PageRank:
                               nodo  pagerank
                     antonio tejero  0.020017
                     alfonso armada  0.013319
                         de defensa  0.009997
                      juan carlos i  0.009625
consejo supremo de justicia militar  0.009302
18:43:53 [ActorGraphCaso] INFO — Figura guardada: figuras/fig2_1_red_principal.png
18:43:54 [ActorGraphCaso] INFO — Figura guardada: figuras/fig2_2_subgrafos_periodo.png
18:43:54 [src.casos.caso2_grafos] INFO — Panel interactivo guardado: figuras/fig2_3_panel_grafo_interactivo.html
18:43:54 [src.casos.caso2_grafos] INFO — CSV guardado: outputs/metricas_centralidad.csv
18:43:54 [src.casos.caso2_grafos] INFO — GEXF gu

=== ActorGraphCaso ===
  n_nodos: 936
  n_aristas: 4223
  n_comunidades: 49
  actor_pagerank_top: antonio tejero
  densidad: 0.0097
  panel_interactivo: figuras/fig2_3_panel_grafo_interactivo.html


## 5. Caso 3 — Topic Modeling

In [7]:
from src.casos.caso3_topics import TopicModelCaso

caso3 = TopicModelCaso(df, output_dir="outputs", fig_dir="figuras", min_topic_size=3)
results3 = caso3.run()
caso3.export()
print(caso3.summary())

18:43:54 [src.casos.caso3_topics] INFO — BERTopic/sentence-transformers no disponible — usando NMF como fallback.
18:43:54 [src.casos.caso3_topics] INFO — umap-learn no disponible — se usará PCA para visualización 2D.
18:43:54 [src.casos.caso3_topics] INFO — === Caso 3: Topic Modeling ===
18:43:54 [src.casos.caso3_topics] INFO — Ajustando NMF (fallback BERTopic)...
18:43:54 [src.casos.caso3_topics] INFO — NMF: 9 tópicos
18:43:55 [TopicModelCaso] INFO — Figura guardada: figuras/fig3_1_topwords_topico.png
18:43:55 [TopicModelCaso] INFO — Figura guardada: figuras/fig3_2_topicos_periodo.png
18:43:55 [TopicModelCaso] INFO — Figura guardada: figuras/fig3_3_scatter_topicos.png
18:43:55 [src.casos.caso3_topics] INFO — Panel interactivo guardado: figuras/fig3_4_panel_topicos_interactivo.html
18:43:55 [TopicModelCaso] INFO — CSV guardado: outputs/topicos_por_documento.csv
18:43:55 [TopicModelCaso] INFO — CSV guardado: outputs/resumen_topicos_caso3.csv
18:43:55 [TopicModelCaso] INFO — CSV guardad

=== TopicModelCaso ===
  backend: NMF
  n_topicos: 9
  n_documentos: 167
  n_outliers: 10
  ari_vs_caso1: N/A
  topico_mayor: 5
  panel_interactivo: figuras/fig3_4_panel_topicos_interactivo.html


## 6. Caso 4 — Clasificador Supervisado

In [8]:
from src.casos.caso4_clasificador import ClasificadorCaso

caso4 = ClasificadorCaso(df, output_dir="outputs", fig_dir="figuras")
results4 = caso4.run()
caso4.export()
print(caso4.summary())

18:43:55 [src.casos.caso4_clasificador] INFO — === Caso 4: Clasificador Supervisado ===
18:43:55 [src.casos.caso4_clasificador] INFO — Distribución de clases:
fuente
Defensa       81
Interior      59
Exteriores    15
18:43:56 [src.casos.caso4_clasificador] INFO — LogisticRegression — F1 macro CV: 0.646 ± 0.078 | Accuracy: 0.639
18:43:58 [src.casos.caso4_clasificador] INFO — RandomForest — F1 macro CV: 0.631 ± 0.112 | Accuracy: 0.697
18:43:59 [src.casos.caso4_clasificador] INFO — LinearSVC — F1 macro CV: 0.652 ± 0.124 | Accuracy: 0.690
18:43:59 [src.casos.caso4_clasificador] INFO — Mejor modelo: LinearSVC (F1 macro CV=0.652)
18:44:00 [src.casos.caso4_clasificador] INFO — Panel interactivo guardado: figuras/fig4_4_panel_clasificador_interactivo.html
18:44:00 [ClasificadorCaso] INFO — Figura guardada: figuras/fig4_1_comparativa_modelos.png
18:44:01 [ClasificadorCaso] INFO — Figura guardada: figuras/fig4_2_confusion_matrix.png
18:44:01 [ClasificadorCaso] INFO — Figura guardada: figuras/fig

=== ClasificadorCaso ===
  mejor_modelo: LinearSVC
  f1_macro_cv: 0.6519
  f1_macro_oof: 0.66
  accuracy_oof: 0.6903
  clases: ['Defensa', 'Exteriores', 'Interior']
  n_train: 155
  n_errores: 48
  panel_interactivo: figuras/fig4_4_panel_clasificador_interactivo.html


## 7. Caso 5 — Análisis Espacio-Temporal

In [9]:
from src.casos.caso5_spatiotemporal import SpatioTemporalCaso

caso5 = SpatioTemporalCaso(df, output_dir="outputs", fig_dir="figuras")
results5 = caso5.run()
caso5.export()
print(caso5.summary())

18:44:01 [src.casos.caso5_spatiotemporal] INFO — === Caso 5: Análisis Espacio-Temporal ===
18:44:01 [SpatioTemporalCaso] INFO — Figura guardada: figuras/fig5_1_serie_temporal.png
18:44:01 [src.casos.caso5_spatiotemporal] INFO — IsolationForest: 17 anomalías detectadas (de 167)
18:44:02 [SpatioTemporalCaso] INFO — Figura guardada: figuras/fig5_2_anomalias_pca.png
18:44:02 [src.casos.caso5_spatiotemporal] INFO — Lugares geocodificados: 21 de 808 únicos
18:44:02 [SpatioTemporalCaso] INFO — Figura guardada: figuras/fig5_3_mapa_geografico.png
18:44:02 [src.casos.caso5_spatiotemporal] INFO — Mapa interactivo guardado: figuras/fig5_3_mapa_geografico_interactivo.html
18:44:02 [SpatioTemporalCaso] INFO — Figura guardada: figuras/fig5_4_heatmap_lugar_periodo.png
18:44:02 [SpatioTemporalCaso] INFO — CSV guardado: outputs/anomalias_caso5.csv
18:44:02 [SpatioTemporalCaso] INFO — CSV guardado: outputs/frecuencia_lugares.csv


=== SpatioTemporalCaso ===
  n_anomalias: 17
  n_lugares_geocodificados: 21
  lugar_mas_frecuente: madrid
  mapa_interactivo: figuras/fig5_3_mapa_geografico_interactivo.html


## 8. Resumen Global

In [10]:
import json

resumen_global = {
    "Caso 1 — EDA": results1,
    "Caso 2 — Grafos": results2,
    "Caso 3 — Topics": results3,
    "Caso 4 — Clasificador": results4,
    "Caso 5 — SpatioTemporal": results5,
}

Path("outputs").mkdir(exist_ok=True)
with open("outputs/resumen_global.json", "w", encoding="utf-8") as f:
    json.dump(resumen_global, f, ensure_ascii=False, indent=2, default=str)

print(json.dumps(resumen_global, indent=2, default=str, ensure_ascii=False))

{
  "Caso 1 — EDA": {
    "total_documentos": 167,
    "documentos_con_ocr": 167,
    "total_palabras": 346599,
    "ministerio_predominante": "Defensa",
    "periodo_dominante": "Desconocido",
    "persona_top": "No Consta",
    "lugar_top": "Madrid",
    "clusters_k": 7
  },
  "Caso 2 — Grafos": {
    "n_nodos": 936,
    "n_aristas": 4223,
    "n_comunidades": 49,
    "actor_pagerank_top": "antonio tejero",
    "densidad": 0.0097,
    "panel_interactivo": "figuras/fig2_3_panel_grafo_interactivo.html"
  },
  "Caso 3 — Topics": {
    "backend": "NMF",
    "n_topicos": 9,
    "n_documentos": 167,
    "n_outliers": 10,
    "ari_vs_caso1": "N/A",
    "topico_mayor": 5,
    "panel_interactivo": "figuras/fig3_4_panel_topicos_interactivo.html"
  },
  "Caso 4 — Clasificador": {
    "mejor_modelo": "LinearSVC",
    "f1_macro_cv": 0.6519,
    "f1_macro_oof": 0.66,
    "accuracy_oof": 0.6903,
    "clases": [
      "Defensa",
      "Exteriores",
      "Interior"
    ],
    "n_train": 155,
    "n_